# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [10]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [11]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

'../data/beir_datasets\\scifact'

In [12]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [13]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [14]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [15]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [16]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [20]:
%pip install rank_bm25
# Parte 2: Retrieval inicial con BM25
from rank_bm25 import BM25Okapi
import numpy as np
import math

# Preparar documentos y queries

doc_ids = list(corpus.keys())
docs = [
    corpus[_id]["text"] if "text" in corpus[_id]
    else corpus[_id].get("title", "") + " " + corpus[_id].get("text", "")
    for _id in doc_ids
]

def simple_tokenize(text):
    return text.lower().split()

tokenized_corpus = [simple_tokenize(d) for d in docs]
bm25 = BM25Okapi(tokenized_corpus)

# Recuperar top-k candidatos por query
TOP_K = 100
bm25_results = {}
for qid, qtext in queries.items():
    tokens = simple_tokenize(qtext)
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    bm25_results[qid] = [(doc_ids[i], float(scores[i])) for i in top_indices]

# Alias con el formato que usa la Parte 3
retrieval_results = {qid: dict(scores) for qid, scores in bm25_results.items()}

# Función de evaluación para Recall@K y DCG@K
def evaluate_baseline(qrels, results, k=10):
    total_recall = 0.0
    total_dcg = 0.0
    valid_queries = 0

    for qid, relevant_docs in qrels.items():
        total_rels = sum(1 for doc_id, rel in relevant_docs.items() if rel > 0)
        if total_rels == 0:
            continue

        valid_queries += 1
        retrieved_docs = results.get(qid, {})
        sorted_retrieved = sorted(retrieved_docs.items(), key=lambda x: x[1], reverse=True)[:k]

        hits = sum(1 for doc_id, _ in sorted_retrieved if doc_id in relevant_docs and relevant_docs[doc_id] > 0)
        total_recall += hits / total_rels

        dcg = 0.0
        for i, (doc_id, _) in enumerate(sorted_retrieved):
            rel = relevant_docs.get(doc_id, 0)
            if rel > 0:
                dcg += rel / math.log2(i + 2)

        total_dcg += dcg

    return total_recall / valid_queries, total_dcg / valid_queries

recall_10, dcg_10 = evaluate_baseline(qrels, retrieval_results, k=10)

print("Métricas del Baseline (BM25)")
print(f"Recall@10: {recall_10:.4f}")
print(f"DCG@10:    {dcg_10:.4f}")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\pc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Métricas del Baseline (BM25)
Recall@10: 0.6688
DCG@10:    0.5744


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [28]:
# Parte 3. Implementación del re-ranking cross-encoder
from sentence_transformers import CrossEncoder
import pandas as pd
from IPython.display import HTML, display

model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(model_name, max_length=512)

reranked_results = {}
position_changes = []

print("Iniciando el re-ranking con Cross-Encoder...")

TOP_K_CE = 10
for qid, bm25_docs in retrieval_results.items():
    query_text = df_queries.loc[df_queries["query_id"] == qid, "query"].values[0]

    top_doc_ids = list(bm25_docs.keys())[:TOP_K_CE]

    pairs = []
    for doc_id in top_doc_ids:
        doc_data = corpus.get(doc_id, {})
        doc_text = doc_data.get("title", "") + " " + doc_data.get("text", "")
        pairs.append([query_text, doc_text])

    if not pairs:
        reranked_results[qid] = {}
        continue

    ce_scores = cross_encoder.predict(pairs, batch_size=8)

    doc_score_ranks = [
        (doc_id, float(score), bm25_rank)
        for bm25_rank, (doc_id, score) in enumerate(zip(top_doc_ids, ce_scores), start=1)
    ]

    doc_score_ranks.sort(key=lambda x: x[1], reverse=True)
    reranked_results[qid] = {doc_id: score for doc_id, score, _ in doc_score_ranks}

    for new_ce_rank, (doc_id, score, old_bm25_rank) in enumerate(doc_score_ranks, start=1):
        if old_bm25_rank != new_ce_rank:
            shift = old_bm25_rank - new_ce_rank
            position_changes.append({
                "query_id": qid,
                "doc_id": doc_id,
                "bm25_rank": old_bm25_rank,
                "ce_rank": new_ce_rank,
                "posiciones_escaladas": shift
            })

print("Re-ranking completado con exito.")

# Identificar qué documentos cambiaron de posición en el top 10
df_changes = pd.DataFrame(position_changes)

if not df_changes.empty:
    print("Resumen de documentos que cambiaron de posicion tras el re-ranking:")
    df_mejorados = df_changes.sort_values(by="posiciones_escaladas", ascending=False).head(15).copy()
    df_mejorados["movimiento"] = df_mejorados["posiciones_escaladas"].apply(
        lambda x: f"+{x}" if x > 0 else str(x)
    )
    df_mejorados = df_mejorados[["query_id", "doc_id", "bm25_rank", "ce_rank", "posiciones_escaladas", "movimiento"]]

    def highlight_shift(value):
        if value > 0:
            return "background-color: #dcfce7; color: #166534; font-weight: 700;"
        if value < 0:
            return "background-color: #fee2e2; color: #991b1b; font-weight: 700;"
        return "background-color: #f3f4f6; color: #374151; font-weight: 700;"

    styled = (
        df_mejorados.style
        .format({
            "bm25_rank": "{:.0f}",
            "ce_rank": "{:.0f}",
            "posiciones_escaladas": "+{:.0f}",
        })
        .hide(axis="index")
        .set_properties(**{"text-align": "center", "padding": "8px 12px"})
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#0f172a"), ("color", "white"), ("font-weight", "700"), ("text-align", "center"), ("padding", "10px 12px")]},
            {"selector": "td", "props": [("border", "1px solid #d1d5db")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%"), ("font-family", "Arial, sans-serif"), ("font-size", "14px")]}
        ])
        .map(highlight_shift, subset=["posiciones_escaladas"])
    )
    display(HTML("<h3 style='margin:0 0 10px 0;color:#0f172a;'>Top documentos que más cambiaron de posición</h3>"))
    display(styled)
else:
    display(HTML("<h3 style='color:#0f172a;'>Los rankings de BM25 y Cross-Encoder fueron idénticos para el Top-10</h3>"))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Iniciando el re-ranking con Cross-Encoder...
Re-ranking completado con exito.
Resumen de documentos que cambiaron de posicion tras el re-ranking:


query_id,doc_id,bm25_rank,ce_rank,posiciones_escaladas,movimiento
887,9641846,10,1,+9,+9
179,16322674,10,1,+9,+9
575,8247597,10,1,+9,+9
1363,25175997,10,2,+8,+8
1202,21258863,10,2,+8,+8
847,4424888,10,2,+8,+8
70,11903247,9,1,+8,+8
72,8317408,10,2,+8,+8
783,854417,10,2,+8,+8
1282,21009874,10,2,+8,+8


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [35]:
# Parte 4: Re-ranking con LTR
from sklearn.ensemble import RandomForestRegressor
from IPython.display import HTML, display
import pandas as pd
import math

# Re-rankear los top-k candidatos para cada query y comparar el Top-10
TOP_K_LTR = 10
ltr_features = []

print("Extrayendo características para LTR...")
for qid, bm25_docs in bm25_results.items():
    query_text = df_queries.loc[df_queries["query_id"] == qid, "query"].values[0].lower()
    q_tokens = set(query_text.split())
    q_len = len(q_tokens)

    top_docs = bm25_docs[:TOP_K_LTR]
    for doc_id, bm25_score in top_docs:
        doc_data = corpus.get(doc_id, {})
        doc_text = (doc_data.get("title", "") + " " + doc_data.get("text", "")).lower()
        d_tokens = set(doc_text.split())
        d_len = len(d_tokens)
        term_overlap = len(q_tokens.intersection(d_tokens))
        relevance = qrels.get(qid, {}).get(doc_id, 0)

        ltr_features.append({
            "query_id": qid,
            "doc_id": doc_id,
            "bm25_score": float(bm25_score),
            "doc_length": d_len,
            "query_length": q_len,
            "term_overlap": term_overlap,
            "relevance": relevance,
        })

df_ltr = pd.DataFrame(ltr_features)

# Entrenamiento del modelo LTR
features = ["bm25_score", "doc_length", "query_length", "term_overlap"]
X = df_ltr[features]
y = df_ltr["relevance"]
ranker = RandomForestRegressor(n_estimators=100, random_state=42)

print("Entrenando el modelo LTR...")
ranker.fit(X, y)

df_ltr["ltr_score"] = ranker.predict(X)

ltr_reranked_results = {}
ltr_position_changes = []

print("Calculando los nuevos rankings...")
for qid, group in df_ltr.groupby("query_id"):
    original_order = group.sort_values(by="bm25_score", ascending=False)["doc_id"].tolist()
    new_order = group.sort_values(by="ltr_score", ascending=False)

    ltr_reranked_results[qid] = {}
    for new_rank, (_, row) in enumerate(new_order.iterrows(), start=1):
        doc_id = row["doc_id"]
        ltr_reranked_results[qid][doc_id] = row["ltr_score"]
        old_rank = original_order.index(doc_id) + 1
        if old_rank != new_rank:
            ltr_position_changes.append({
                "query_id": qid,
                "doc_id": doc_id,
                "bm25_rank": old_rank,
                "ltr_rank": new_rank,
                "posiciones_escaladas": old_rank - new_rank,
            })

# Mostrar la tabla de cambios
print("\nResumen de documentos que cambiaron de posicion tras el LTR:")
df_ltr_changes = pd.DataFrame(ltr_position_changes)
if not df_ltr_changes.empty:
    df_mejorados_ltr = df_ltr_changes.sort_values(by="posiciones_escaladas", ascending=False).head(15).copy()
    df_mejorados_ltr["movimiento"] = df_mejorados_ltr["posiciones_escaladas"].apply(lambda x: f"+{x}" if x > 0 else str(x))
    df_mejorados_ltr = df_mejorados_ltr[["query_id", "doc_id", "bm25_rank", "ltr_rank", "movimiento"]]

    styled_ltr = (
        df_mejorados_ltr.style
        .hide(axis="index")
        .set_properties(**{"text-align": "center", "padding": "8px 12px"})
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#0f172a"), ("color", "white"), ("font-weight", "700"), ("text-align", "center")]},
            {"selector": "td", "props": [("border", "1px solid #d1d5db")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%"), ("font-family", "Arial, sans-serif")]},
        ])
    )
    display(HTML("<h3 style='margin:0 0 10px 0;color:#0f172a;'>Documentos que cambiaron de posición en el Top-10</h3>"))
    display(styled_ltr)
else:
    display(HTML("<h3 style='color:#0f172a;'>Los rankings de BM25 y LTR fueron idénticos para el Top-10</h3>"))


def evaluate_ltr_metrics(qrels, results, k=10):
    total_recall = 0.0
    total_dcg = 0.0
    total_ap = 0.0
    valid_queries = 0

    for qid, relevant_docs in qrels.items():
        total_rels = sum(1 for doc_id, rel in relevant_docs.items() if rel > 0)
        if total_rels == 0:
            continue

        valid_queries += 1
        retrieved_docs = results.get(qid, {})
        sorted_retrieved = sorted(retrieved_docs.items(), key=lambda x: x[1], reverse=True)[:k]

        hits = 0
        dcg = 0.0
        sum_precisions = 0.0

        for i, (doc_id, _) in enumerate(sorted_retrieved):
            rel = relevant_docs.get(doc_id, 0)
            if rel > 0:
                hits += 1
                dcg += rel / math.log2(i + 2)
                sum_precisions += hits / (i + 1)

        total_recall += hits / total_rels
        total_dcg += dcg
        total_ap += sum_precisions / total_rels

    return {
        f"Recall@{k}": total_recall / valid_queries if valid_queries else float('nan'),
        f"DCG@{k}": total_dcg / valid_queries if valid_queries else float('nan'),
        "MAP": total_ap / valid_queries if valid_queries else float('nan'),
    }

ltr_metrics = evaluate_ltr_metrics(qrels, ltr_reranked_results, k=10)
ltr_metrics

Extrayendo características para LTR...
Entrenando el modelo LTR...
Calculando los nuevos rankings...

Resumen de documentos que cambiaron de posicion tras el LTR:


query_id,doc_id,bm25_rank,ltr_rank,movimiento
1278,40632104,10,1,+9
1282,21009874,10,1,+9
133,17934082,10,1,+9
544,2638387,10,1,+9
179,16322674,10,1,+9
294,13613916,10,1,+9
636,16346504,10,2,+8
907,13235609,10,2,+8
478,23801039,9,1,+8
127,15588516,10,2,+8


{'Recall@10': 0.6688333333333334,
 'DCG@10': 0.7183394453364361,
 'MAP': 0.6688333333333334}

## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [37]:
# Parte 5: Evaluación post re-ranking
import math

# Asegurar formato consistente para evaluar
cross_results_for_eval = {
    qid: sorted(docs.items(), key=lambda x: x[1], reverse=True)
    for qid, docs in reranked_results.items()
}

ltr_results_for_eval = {
    qid: sorted(docs.items(), key=lambda x: x[1], reverse=True)
    for qid, docs in ltr_reranked_results.items()
}


def mean_average_precision(results, qrels, k=10):
    ap_values = []
    for qid, relevant_docs in qrels.items():
        rel_set = {doc_id for doc_id, rel in relevant_docs.items() if rel > 0}
        if not rel_set:
            continue

        ranked_docs = results.get(qid, [])[:k]
        hits = 0
        precision_sum = 0.0
        for rank, (doc_id, _) in enumerate(ranked_docs, start=1):
            if doc_id in rel_set:
                hits += 1
                precision_sum += hits / rank
        ap_values.append(precision_sum / len(rel_set))

    return sum(ap_values) / len(ap_values) if ap_values else float('nan')


def recall_and_ndcg(results, qrels, k=10):
    recalls = []
    ndcgs = []
    for qid, relevant_docs in qrels.items():
        rel_docs = {doc_id for doc_id, rel in relevant_docs.items() if rel > 0}
        if not rel_docs:
            continue

        ranked_docs = results.get(qid, [])[:k]
        retrieved_ids = [doc_id for doc_id, _ in ranked_docs]
        hits = sum(1 for doc_id in retrieved_ids if doc_id in rel_docs)
        recalls.append(hits / len(rel_docs))

        dcg = 0.0
        for i, (doc_id, _) in enumerate(ranked_docs):
            rel = relevant_docs.get(doc_id, 0)
            if rel > 0:
                dcg += (2 ** rel - 1) / math.log2(i + 2)

        ideal_rels = sorted((rel for rel in relevant_docs.values() if rel > 0), reverse=True)[:k]
        idcg = 0.0
        for i, rel in enumerate(ideal_rels):
            idcg += (2 ** rel - 1) / math.log2(i + 2)

        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return (
        sum(recalls) / len(recalls) if recalls else float('nan'),
        sum(ndcgs) / len(ndcgs) if ndcgs else float('nan'),
    )


bm25_r10, bm25_ndcg10 = recall_and_ndcg(bm25_results, qrels, k=10)
bm25_map10 = mean_average_precision(bm25_results, qrels, k=10)

ce_r10, ce_ndcg10 = recall_and_ndcg(cross_results_for_eval, qrels, k=10)
ce_map10 = mean_average_precision(cross_results_for_eval, qrels, k=10)

ltr_r10, ltr_ndcg10 = recall_and_ndcg(ltr_results_for_eval, qrels, k=10)
ltr_map10 = mean_average_precision(ltr_results_for_eval, qrels, k=10)

metrics_table = pd.DataFrame([
    {"Modelo": "1. Baseline (BM25)", "Recall@10": bm25_r10, "nDCG@10": bm25_ndcg10, "MAP": bm25_map10},
    {"Modelo": "2. Re-ranking: Cross-Encoder", "Recall@10": ce_r10, "nDCG@10": ce_ndcg10, "MAP": ce_map10},
    {"Modelo": "3. Re-ranking: LTR", "Recall@10": ltr_r10, "nDCG@10": ltr_ndcg10, "MAP": ltr_map10},
])

print("Comparativa final de métricas:")
display(metrics_table.round(4))

example_qid = '133' if '133' in queries else list(queries.keys())[0]
print('\nQuery:', queries[example_qid])
print('\nTop-10 BM25:')
for doc_id, score in bm25_results[example_qid][:10]:
    print(doc_id, 'score=', score)

print('\nTop-10 Cross-encoder:')
for doc_id, score in cross_results_for_eval[example_qid][:10]:
    print(doc_id, 'score=', score)

print('\nTop-10 LTR:')
for doc_id, score in ltr_results_for_eval[example_qid][:10]:
    print(doc_id, 'score=', score)

Comparativa final de métricas:


,Modelo,Recall@10,nDCG@10,MAP
0,1. Baseline (BM25),0.6688,0.5438,0.4993
1,2. Re-ranking: Cross-Encoder,0.6688,0.6038,0.5764
2,3. Re-ranking: LTR,0.6688,0.6729,0.6688



Query: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Top-10 BM25:
26688294 score= 55.1964401863664
37964706 score= 50.04011691148892
9507605 score= 50.03740998262752
5270265 score= 45.70320340871325
12785130 score= 45.06239524984214
45764440 score= 45.044936360734006
86694016 score= 44.899837173478296
12640810 score= 44.68953632297576
5821617 score= 44.451933317595774
17934082 score= 44.43642301215712

Top-10 Cross-encoder:
12640810 score= 0.35971733927726746
9507605 score= -2.60658597946167
86694016 score= -2.6976735591888428
17934082 score= -3.794553756713867
37964706 score= -6.668342113494873
45764440 score= -9.50143051147461
12785130 score= -9.506258010864258
5821617 score= -9.593212127685547
26688294 score= -9.898658752441406
5270265 score= -10.446354866027832

Top-10 LTR:
17934082 score= 0.76
12640810 score= 0.73
5270265 score= 0.23
26688294 score= 0.11
5821617 score= 0